In [ ]:
import psutil
import platform
import torch

#Please check you capability before starting, dont brick your system
print("Basic Systen Information :")

ram = psutil.virtual_memory()
print(f"RAM: {ram.total / (1024 ** 3):.2f} GB")

cpu_info = platform.processor()
cpu_count = psutil.cpu_count(logical=True)

print(f"CPU: {cpu_info}")
print(f"Cores: {cpu_count}")

gpu_count = torch.cuda.device_count()
gpu_name = torch.cuda.get_device_name(0)
vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 2)  # Convert bytes to MB

print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_total:.2f} MB")


Basic Systen Information :
RAM: 12.67 GB
CPU: x86_64
Cores: 2
GPU: Tesla T4
VRAM: 15095.06 MB


In [ ]:
#Install basic need
!pip install -q transformers accelerate datasets evaluate torch torchvision tqdm
!pip install -q pennylane pennylane-lightning
!pip install -q peft
!pip install -q huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.1/57.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 81.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 kB 16.5 MB/s eta 0:00:00


In [ ]:
import os, random, json, time
import torch, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
torch.backends.cudnn.benchmark = True

#checkpoint, incase collab runtime ends
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CKPT_DIR = "/content/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)


In [1]:
#Quick re-check GPU, make sure to see Driver and CUDA version is fairly recent
!nvidia-smi

Mon Nov 10 11:21:34 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             12W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# REMINDER : Google Collab storage is ephemeral, it will delete itself. Make sure to download checkpoints!!!!

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, os

MODEL_NAME = "EleutherAI/gpt-neo-125M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()
print("Loaded", MODEL_NAME, "on", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/526M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Loaded EleutherAI/gpt-neo-125M on cuda


In [7]:
for n, p in model.named_parameters():
    p.requires_grad = False
for name, param in model.named_parameters():
    if "lm_head" in name or "transformer.h.11" in name:
        param.requires_grad = True

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total:,}, Trainable: {trainable:,}")


Total params: 125,198,592, Trainable: 7,085,568


In [8]:
import pennylane as qml
from pennylane import numpy as pnp
import torch.nn as nn
import torch

class VQCAdapter(nn.Module):
    def __init__(self, hidden_dim, n_qubits=3, dev_name="default.qubit", device_cpu=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_qubits = n_qubits
        self.vec_dim = 2 ** n_qubits
        self.proj = nn.Linear(hidden_dim, self.vec_dim)
        self.post = nn.Linear(self.vec_dim, hidden_dim)
        self.q_params = torch.randn(self.n_qubits * 3, requires_grad=True)
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.qnode = qml.QNode(self._circuit, self.dev, interface="torch")
        self.to_cpu = device_cpu

    def _circuit(self, inputs, qparams):
        for i in range(self.n_qubits):
            qml.RY(inputs[i], wires=i)
        idx = 0
        for i in range(self.n_qubits):
            qml.RZ(qparams[idx], wires=i); idx += 1
            qml.RY(qparams[idx], wires=i); idx += 1
            qml.RZ(qparams[idx], wires=i); idx += 1
        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

    def forward(self, hidden):  
        x = self.proj(hidden)  
        x = torch.tanh(x)
        outputs = []
        qparams = self.q_params
        for row in x.detach().cpu():
            inp = row[:self.n_qubits].numpy()
            res = torch.tensor(self.qnode(inp, qparams.detach().cpu().numpy()), dtype=torch.float32)
            outputs.append(res)
        out = torch.stack(outputs).to(hidden.device)
        out = self.post(out)
        return out

/usr/local/lib/python3.12/dist-packages/pennylane/__init__.py:209: RuntimeWarning: PennyLane is not yet compatible with JAX versions > 0.6.2. You have version 0.7.2 installed. Please downgrade JAX to 0.6.2 to avoid runtime errors using python -m pip install jax~=0.6.0 jaxlib~=0.6.0
  warnings.warn(


In [10]:
from datasets import load_dataset
from torch.utils.data import DataLoader
import torch.optim as optim

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train[:1%]")
val_ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation[:1%]")

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=64)

dataset = dataset.map(lambda x: tokenize_batch(x), batched=True)
val_ds = val_ds.map(lambda x: tokenize_batch(x), batched=True)

dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask"])

train_loader = DataLoader(dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=8)

model_optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5)
vqc = VQCAdapter(hidden_dim=model.config.hidden_size, n_qubits=3)
vqc_optimizer = optim.AdamW([vqc.q_params], lr=5e-4)
scaler = torch.cuda.amp.GradScaler()

Map:   0%|          | 0/367 [00:00<?, ? examples/s]

Map:   0%|          | 0/38 [00:00<?, ? examples/s]

/tmp/ipython-input-3387197288.py:26: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [29]:
# No-VQC Baseline
import os
import torch
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "EleutherAI/gpt-neo-125M"
model_no_vqc = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_no_vqc.to(device)

for n, p in model_no_vqc.named_parameters():
    p.requires_grad = False
for name, param in model_no_vqc.named_parameters():
    if "lm_head" in name or "transformer.h.11" in name:
        param.requires_grad = True

model_no_vqc.train()
print("Re-loaded", MODEL_NAME, "on", device, "with dtype torch.float32 for No VQC baseline")

scaler_no_vqc = torch.cuda.amp.GradScaler()

model_optimizer_no_vqc = optim.AdamW([p for p in model_no_vqc.parameters() if p.requires_grad], lr=2e-5)

steps_no_vqc = 2000
eval_every_no_vqc = 100
ckpt_every_no_vqc = 1000
step_no_vqc = 0

def evaluate_no_vqc():
    model_no_vqc.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                out = model_no_vqc(input_ids=ids, attention_mask=att, labels=ids)
                loss = out.loss
            total_loss += loss.item()
            count += 1
    model_no_vqc.train()
    return total_loss / max(1, count)


print("Starting training for No VQC baseline...")

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer_no_vqc.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model_no_vqc(input_ids=ids, attention_mask=att, output_hidden_states=True)
            lm_logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = lm_logits[:, :-1, :].contiguous()
        shift_labels = ids[:, 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        scaler_no_vqc.scale(loss).backward()
        scaler_no_vqc.step(model_optimizer_no_vqc)
        scaler_no_vqc.update()

        step_no_vqc += 1
        if step_no_vqc % eval_every_no_vqc == 0:
            val_loss_no_vqc = evaluate_no_vqc()
            print(f"No VQC Baseline Step {step_no_vqc} Eval Loss: {val_loss_no_vqc:.4f}")
        if step_no_vqc >= steps_no_vqc:
            break
    if step_no_vqc >= steps_no_vqc:
        break

val_loss_no_vqc = evaluate_no_vqc()
print(f"No VQC Baseline Training finished at step {step_no_vqc}. Final val loss: {val_loss_no_vqc:.4f}")

Re-loaded EleutherAI/gpt-neo-125M on cuda with dtype torch.float32 for No VQC baseline
Starting training for No VQC baseline...


/tmp/ipython-input-688828048.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_no_vqc = torch.cuda.amp.GradScaler()


No VQC Baseline Step 100 Eval Loss: 2.0271
No VQC Baseline Step 200 Eval Loss: 1.9638
No VQC Baseline Step 300 Eval Loss: 1.9299
No VQC Baseline Step 400 Eval Loss: 1.9078
No VQC Baseline Step 500 Eval Loss: 1.8914
No VQC Baseline Step 600 Eval Loss: 1.8774
No VQC Baseline Step 700 Eval Loss: 1.8696
No VQC Baseline Step 800 Eval Loss: 1.8598
No VQC Baseline Step 900 Eval Loss: 1.8574
No VQC Baseline Step 1000 Eval Loss: 1.8511
No VQC Baseline Step 1100 Eval Loss: 1.8498
No VQC Baseline Step 1200 Eval Loss: 1.8488
No VQC Baseline Step 1300 Eval Loss: 1.8463
No VQC Baseline Step 1400 Eval Loss: 1.8448
No VQC Baseline Step 1500 Eval Loss: 1.8474
No VQC Baseline Step 1600 Eval Loss: 1.8467
No VQC Baseline Step 1700 Eval Loss: 1.8465
No VQC Baseline Step 1800 Eval Loss: 1.8518
No VQC Baseline Step 1900 Eval Loss: 1.8516
No VQC Baseline Step 2000 Eval Loss: 1.8562
No VQC Baseline Training finished at step 2000. Final val loss: 1.8562


In [30]:
# LoRA Baseline Training

import os
import torch
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_NAME = "EleutherAI/gpt-neo-125M"
model_lora = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_lora.to(device)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM", 
)

model_lora = get_peft_model(model_lora, lora_config)

model_lora.print_trainable_parameters()

model_lora.train() 

print("Loaded", MODEL_NAME, "on", device, "with LoRA adapter")

scaler_lora = torch.cuda.amp.GradScaler()

model_optimizer_lora = optim.AdamW(model_lora.parameters(), lr=2e-5)

steps_lora = 2000 
eval_every_lora = 100
ckpt_every_lora = 1000 
step_lora = 0

def evaluate_lora():
    model_lora.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                out = model_lora(input_ids=ids, attention_mask=att, labels=ids)
                loss = out.loss
            total_loss += loss.item()
            count += 1
    model_lora.train()
    return total_loss / max(1, count)


print("Starting training for LoRA baseline...")

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer_lora.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model_lora(input_ids=ids, attention_mask=att, labels=ids)
            loss = outputs.loss

        scaler_lora.scale(loss).backward()
        scaler_lora.step(model_optimizer_lora)
        scaler_lora.update()

        step_lora += 1
        if step_lora % eval_every_lora == 0:
            val_loss_lora = evaluate_lora()
            print(f"LoRA Baseline Step {step_lora} Eval Loss: {val_loss_lora:.4f}")
        if step_lora >= steps_lora:
            break
    if step_lora >= steps_lora:
        break

val_loss_lora = evaluate_lora()
print(f"LoRA Baseline Training finished at step {step_lora}. Final val loss: {val_loss_lora:.4f}")

trainable params: 294,912 || all params: 125,493,504 || trainable%: 0.2350
Loaded EleutherAI/gpt-neo-125M on cuda with LoRA adapter
Starting training for LoRA baseline...


/tmp/ipython-input-75339369.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_lora = torch.cuda.amp.GradScaler()


LoRA Baseline Step 100 Eval Loss: 6.2692
LoRA Baseline Step 200 Eval Loss: 4.9451
LoRA Baseline Step 300 Eval Loss: 2.4977
LoRA Baseline Step 400 Eval Loss: 2.0634
LoRA Baseline Step 500 Eval Loss: 1.9947
LoRA Baseline Step 600 Eval Loss: 1.9635
LoRA Baseline Step 700 Eval Loss: 1.9432
LoRA Baseline Step 800 Eval Loss: 1.9276
LoRA Baseline Step 900 Eval Loss: 1.9143
LoRA Baseline Step 1000 Eval Loss: 1.9033
LoRA Baseline Step 1100 Eval Loss: 1.8937
LoRA Baseline Step 1200 Eval Loss: 1.8850
LoRA Baseline Step 1300 Eval Loss: 1.8767
LoRA Baseline Step 1400 Eval Loss: 1.8692
LoRA Baseline Step 1500 Eval Loss: 1.8633
LoRA Baseline Step 1600 Eval Loss: 1.8567
LoRA Baseline Step 1700 Eval Loss: 1.8508
LoRA Baseline Step 1800 Eval Loss: 1.8450
LoRA Baseline Step 1900 Eval Loss: 1.8399
LoRA Baseline Step 2000 Eval Loss: 1.8351
LoRA Baseline Training finished at step 2000. Final val loss: 1.8351


In [26]:
import torch
import torch.nn as nn
import pennylane as qml
from pennylane import numpy as pnp

class VQCAdapter(nn.Module):
    def __init__(self, hidden_dim, n_qubits=3, dev_name="default.qubit", device_cpu=True):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_qubits = n_qubits
        self.vec_dim = 2 ** n_qubits
        self.proj = nn.Linear(hidden_dim, self.vec_dim)
        self.post = nn.Linear(self.vec_dim, hidden_dim)
        self.q_params = torch.nn.Parameter(torch.randn(self.n_qubits * 3))
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.qnode = qml.QNode(self._circuit, self.dev, interface="torch")
        self.to_cpu = device_cpu
        self.n_qubits_to_vec_dim = torch.nn.Linear(self.n_qubits, self.vec_dim)


    def _circuit(self, inputs, qparams):
        for i in range(self.n_qubits):
            qml.RY(inputs[i], wires=i)
        idx = 0
        for i in range(self.n_qubits):
            qml.RZ(qparams[idx], wires=i); idx += 1
            qml.RY(qparams[idx], wires=i); idx += 1
            qml.RZ(qparams[idx], wires=i); idx += 1
        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

    def forward(self, hidden):  
        pass


def vqc_forward_cpu(hidden):
    vqc.proj.to("cpu")
    vqc.post.to("cpu")
    vqc.q_params = vqc.q_params.to("cpu") 
    vqc.n_qubits_to_vec_dim.to("cpu")


    hid_cpu = hidden.detach().cpu()
    x = vqc.proj(hid_cpu)                      
    x = torch.tanh(x)
    outputs = []
    qparams = vqc.q_params.detach().cpu()
    for row in x:
        inp = row.view(-1)                       
        pooled = inp[:vqc.n_qubits].detach().numpy()
        res = torch.tensor(vqc.qnode(pooled, qparams.numpy()), dtype=torch.float32)
        outputs.append(res)
    out = torch.stack(outputs)
    mapped_out = vqc.n_qubits_to_vec_dim(out) 
    out = vqc.post(mapped_out)                          
    return out.to(hidden.device)


batch_size = 8
hidden_dim = model.config.hidden_size 
dummy_hidden = torch.randn(batch_size, hidden_dim, dtype=torch.float32).to(device)

vqc = VQCAdapter(hidden_dim=hidden_dim, n_qubits=3)

output = vqc_forward_cpu(dummy_hidden)

print(f"Output shape: {output.shape}")
print(f"Output dtype: {output.dtype}")

Output shape: torch.Size([8, 768])
Output dtype: torch.float32


In [28]:
#VQC-only Model

import os
import torch
from tqdm import tqdm
import torch.optim as optim

def save_checkpoint(step):
    tmp = "/content/checkpoints/tmp_ckpt.pt"
    torch.save({
        "step": step,
        "model_state": model.state_dict(),
        "vqc_state": {"proj": vqc.proj.state_dict(), "post": vqc.post.state_dict(), "q_params": vqc.q_params.detach().cpu(), "n_qubits_to_vec_dim": vqc.n_qubits_to_vec_dim.state_dict()},
        "optimizer_state": model_optimizer.state_dict(),
        "scaler_state": scaler.state_dict(),
    }, tmp)
    final = f"/content/checkpoints/ckpt_{step}.pt"
    os.replace(tmp, final)
    print("ATTENTION : New Checkpoint made, download fast!!!!!")
    files = sorted([f for f in os.listdir("/content/checkpoints") if f.startswith("ckpt_")])
    if len(files) > 3:
        os.remove("/content/checkpoints/" + files[0])

def evaluate():
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"): 
                out = model(input_ids=ids, attention_mask=att, labels=ids)
                loss = out.loss
            total_loss += loss.item()
            count += 1
    model.train()
    return total_loss / max(1, count)

vqc.n_qubits = 3
vqc.vec_dim = 2 ** vqc.n_qubits
vqc.proj = torch.nn.Linear(model.config.hidden_size, vqc.vec_dim).to("cpu")
vqc.post = torch.nn.Linear(vqc.vec_dim, model.config.hidden_size).to("cpu")
vqc.q_params = torch.nn.Parameter(torch.randn(vqc.n_qubits * 3)).to("cpu")
vqc.n_qubits_to_vec_dim = torch.nn.Linear(vqc.n_qubits, vqc.vec_dim).to("cpu")

steps = 2000
eval_every = 100
ckpt_every = 1000
step = 0

model_optimizer = optim.AdamW([
    {'params': [p for p in model.parameters() if p.requires_grad]},
    {'params': vqc.proj.parameters(), 'lr': 5e-4},
    {'params': vqc.post.parameters(), 'lr': 5e-4},
    {'params': vqc.n_qubits_to_vec_dim.parameters(), 'lr': 5e-4}
], lr=2e-5)

scaler = torch.cuda.amp.GradScaler()

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer.zero_grad()

        with torch.amp.autocast("cuda"): 
            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
            hidden = outputs.hidden_states[-1] 
            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
            lm_logits = model.lm_head(hidden)
            loss_fct = torch.nn.CrossEntropyLoss()
            shift_logits = lm_logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        scaler.scale(loss).backward() 
        if vqc.q_params.grad is not None:
             if not torch.isfinite(vqc.q_params.grad).all():
                 print(f"Step {step}: Warning: NaN or Inf gradient found for vqc.q_params. Skipping update.")
                 vqc.q_params.grad = None 
             else:
                with torch.no_grad():
                    vqc.q_params -= 5e-4 * vqc.q_params.grad
                vqc.q_params.grad = None
        scaler.step(model_optimizer) 
        scaler.update()
        step += 1
        if step % eval_every == 0:
            val_loss = evaluate()
            print(f"Step {step} Eval Loss: {val_loss:.4f}")
        if step % ckpt_every == 0:
            save_checkpoint(step)
        if step >= steps:
            save_checkpoint(step)
            break
    if step >= steps:
        break
val_loss = evaluate()
print(f"Training finished at step {step}. Final val loss: {val_loss:.4f}")
save_checkpoint(step)

/tmp/ipython-input-1998562110.py:62: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Step 100 Eval Loss: 1.9309
Step 200 Eval Loss: 1.9407
Step 300 Eval Loss: 1.9389
Step 400 Eval Loss: 1.9436
Step 500 Eval Loss: 1.9523
Step 600 Eval Loss: 1.9596
Step 700 Eval Loss: 1.9619
Step 800 Eval Loss: 1.9699
Step 900 Eval Loss: 1.9767
Step 1000 Eval Loss: 1.9812
ATTENTION : New Checkpoint made, download fast!!!!!
Step 1100 Eval Loss: 1.9872
Step 1200 Eval Loss: 1.9919
Step 1300 Eval Loss: 2.0035
Step 1400 Eval Loss: 2.0074
Step 1500 Eval Loss: 2.0142
Step 1600 Eval Loss: 2.0238
Step 1700 Eval Loss: 2.0333
Step 1800 Eval Loss: 2.0397
Step 1900 Eval Loss: 2.0505
Step 2000 Eval Loss: 2.0490
ATTENTION : New Checkpoint made, download fast!!!!!
ATTENTION : New Checkpoint made, download fast!!!!!
Training finished at step 2000. Final val loss: 2.0490
ATTENTION : New Checkpoint made, download fast!!!!!


In [33]:
#Combined LoRA and VQC Model (3 Qubits, Original Circuit, VQC LR 5e-4)
import os
import torch
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

MODEL_NAME = "EleutherAI/gpt-neo-125M"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, 
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

model.train() 
print("Loaded", MODEL_NAME, "on", device, "with LoRA adapter")

hidden_dim = model.config.hidden_size
vqc = VQCAdapter(hidden_dim=hidden_dim, n_qubits=3)
vqc.n_qubits = 3 
vqc.vec_dim = 2 ** vqc.n_qubits
vqc.proj = torch.nn.Linear(hidden_dim, vqc.vec_dim).to("cpu")
vqc.post = torch.nn.Linear(vqc.vec_dim, hidden_dim).to("cpu")

vqc.q_params = torch.nn.Parameter(torch.randn(vqc.n_qubits * 3)).to("cpu")
vqc.n_qubits_to_vec_dim = torch.nn.Linear(vqc.n_qubits, vqc.vec_dim).to("cpu")
print(f"Instantiated VQCAdapter with hidden_dim={hidden_dim}, n_qubits={vqc.n_qubits} on CPU")


def vqc_forward_cpu(hidden):
    vqc.proj.to("cpu")
    vqc.post.to("cpu")
    vqc.q_params = vqc.q_params.to("cpu") 
    vqc.n_qubits_to_vec_dim.to("cpu")

    hid_cpu = hidden.detach().cpu()
    x = vqc.proj(hid_cpu)                     
    x = torch.tanh(x)
    outputs = []
    qparams = vqc.q_params.detach().cpu()
    for row in x:
        inp = row.view(-1)                      
        pooled = inp[:vqc.n_qubits].detach().numpy()
        res = torch.tensor(vqc.qnode(pooled, qparams.numpy()), dtype=torch.float32)
        outputs.append(res)
    out = torch.stack(outputs)                   
    mapped_out = vqc.n_qubits_to_vec_dim(out) 
    out = vqc.post(mapped_out)                         
    return out.to(hidden.device)
    
scaler = torch.cuda.amp.GradScaler()

steps = 2000
eval_every = 100
ckpt_every = 1000
step = 0

model_optimizer = optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and not "lora_" in n]}, 
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n], 'lr': 2e-5}, 
    {'params': vqc.proj.parameters(), 'lr': 5e-4},
    {'params': vqc.post.parameters(), 'lr': 5e-4},
    {'params': vqc.n_qubits_to_vec_dim.parameters(), 'lr': 5e-4}
], lr=2e-5)


print("Starting training for Combined LoRA and VQC model...")

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
            hidden = outputs.hidden_states[-1] 
            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
            lm_logits = model.lm_head(hidden)
            loss_fct = torch.nn.CrossEntropyLoss()
            shift_logits = lm_logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        scaler.scale(loss).backward()

        if vqc.q_params.grad is not None:
             if not torch.isfinite(vqc.q_params.grad).all():
                 print(f"Step {step}: Warning: NaN or Inf gradient found for vqc.q_params. Skipping update.")
                 vqc.q_params.grad = None
             else:
                with torch.no_grad():
                    vqc.q_params -= 5e-4 * vqc.q_params.grad
                vqc.q_params.grad = None

        scaler.step(model_optimizer)
        scaler.update()


        step += 1
        if step % eval_every == 0:
            def evaluate_combined():
                model.eval() 
                total_loss = 0.0
                count = 0
                with torch.no_grad():
                    for batch in val_loader:
                        ids = batch["input_ids"].to(device)
                        att = batch["attention_mask"].to(device)
                        with torch.amp.autocast("cuda"):
                            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                            hidden = outputs.hidden_states[-1]
                            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                            lm_logits = model.lm_head(hidden)
                            loss_fct_eval = torch.nn.CrossEntropyLoss()
                            shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                            shift_labels_eval = ids[:, 1:].contiguous()
                            loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))

                        total_loss += loss.item()
                        count += 1
                model.train()
                return total_loss / max(1, count)

            val_loss_combined = evaluate_combined()
            print(f"Combined Model Step {step} Eval Loss: {val_loss_combined:.4f}")

        if step >= steps:
            break
    if step >= steps:
        break

def evaluate_combined():
    model.eval() 
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                hidden = outputs.hidden_states[-1]
                vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                lm_logits = model.lm_head(hidden)
                loss_fct_eval = torch.nn.CrossEntropyLoss()
                shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                shift_labels_eval = ids[:, 1:].contiguous()
                loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
            total_loss += loss.item()
            count += 1
    model.train()
    return total_loss / max(1, count)

val_loss_combined = evaluate_combined()
print(f"Combined LoRA and VQC Training finished at step {step}. Final val loss: {val_loss_combined:.4f}")

trainable params: 294,912 || all params: 125,493,504 || trainable%: 0.2350
Loaded EleutherAI/gpt-neo-125M on cuda with LoRA adapter
Instantiated VQCAdapter with hidden_dim=768, n_qubits=3 on CPU
Starting training for Combined LoRA and VQC model...


/tmp/ipython-input-2988348527.py:79: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Combined Model Step 100 Eval Loss: 6.2523
Combined Model Step 200 Eval Loss: 4.9298
Combined Model Step 300 Eval Loss: 2.4043
Combined Model Step 400 Eval Loss: 2.0632
Combined Model Step 500 Eval Loss: 1.9932
Combined Model Step 600 Eval Loss: 1.9622
Combined Model Step 700 Eval Loss: 1.9426
Combined Model Step 800 Eval Loss: 1.9270
Combined Model Step 900 Eval Loss: 1.9143
Combined Model Step 1000 Eval Loss: 1.9033
Combined Model Step 1100 Eval Loss: 1.8937
Combined Model Step 1200 Eval Loss: 1.8853
Combined Model Step 1300 Eval Loss: 1.8772
Combined Model Step 1400 Eval Loss: 1.8700
Combined Model Step 1500 Eval Loss: 1.8629
Combined Model Step 1600 Eval Loss: 1.8567
Combined Model Step 1700 Eval Loss: 1.8511
Combined Model Step 1800 Eval Loss: 1.8457
Combined Model Step 1900 Eval Loss: 1.8403
Combined Model Step 2000 Eval Loss: 1.8352
Combined LoRA and VQC Training finished at step 2000. Final val loss: 1.8352


In [39]:
#Combined LoRA and VQC Model (4 Qubits, Original Circuit, VQC LR 5e-4)

import os
import torch
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import pennylane as qml
from pennylane import numpy as pnp
import torch.nn as nn

class VQCAdapter(nn.Module):
    def __init__(self, hidden_dim, n_qubits=4, dev_name="default.qubit", device_cpu=True): 
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_qubits = n_qubits
        self.vec_dim = 2 ** self.n_qubits 
        self.proj = nn.Linear(hidden_dim, self.vec_dim)
        self.post = nn.Linear(self.vec_dim, hidden_dim)
        self.q_params = torch.nn.Parameter(torch.randn(self.n_qubits * 3))
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.qnode = qml.QNode(self._circuit, self.dev, interface="torch")
        self.to_cpu = device_cpu
        self.n_qubits_to_vec_dim = torch.nn.Linear(self.n_qubits, self.vec_dim)

    def _circuit(self, inputs, qparams):
        for i in range(self.n_qubits):
            qml.RY(inputs[i], wires=i)
        idx = 0
        for i in range(self.n_qubits):
            qml.RZ(qparams[idx], wires=i); idx += 1
            qml.RY(qparams[idx], wires=i); idx += 1
            qml.RZ(qparams[idx], wires=i); idx += 1
        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

    def forward(self, hidden):
        pass 

def vqc_forward_cpu(hidden):
    vqc.proj.to("cpu")
    vqc.post.to("cpu")
    vqc.q_params = vqc.q_params.to("cpu") 
    vqc.n_qubits_to_vec_dim.to("cpu")

    hid_cpu = hidden.detach().cpu()
    x = vqc.proj(hid_cpu)                       
    x = torch.tanh(x)
    outputs = []
    qparams = vqc.q_params.detach().cpu()
    for row in x:
        inp = row.view(-1)                       
        pooled = inp[:vqc.n_qubits].detach().numpy()
        res = torch.tensor(vqc.qnode(pooled, qparams.numpy()), dtype=torch.float32)
        outputs.append(res)
    out = torch.stack(outputs)                   
    mapped_out = vqc.n_qubits_to_vec_dim(out) 
    out = vqc.post(mapped_out)                       
    return out.to(hidden.device) 

MODEL_NAME = "EleutherAI/gpt-neo-125M"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM", 
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

model.train() 
print("Loaded", MODEL_NAME, "on", device, "with LoRA adapter")

hidden_dim = model.config.hidden_size
vqc = VQCAdapter(hidden_dim=hidden_dim, n_qubits=4)
vqc.n_qubits = 4 
vqc.vec_dim = 2 ** vqc.n_qubits 
vqc.proj = torch.nn.Linear(hidden_dim, vqc.vec_dim).to("cpu")
vqc.post = torch.nn.Linear(vqc.vec_dim, hidden_dim).to("cpu")

vqc.q_params = torch.nn.Parameter(torch.randn(vqc.n_qubits * 3)).to("cpu")

vqc.n_qubits_to_vec_dim = torch.nn.Linear(vqc.n_qubits, vqc.vec_dim).to("cpu")
print(f"Instantiated VQCAdapter with hidden_dim={hidden_dim}, n_qubits={vqc.n_qubits}, vec_dim={vqc.vec_dim} on CPU")

scaler = torch.cuda.amp.GradScaler()

steps = 2000
eval_every = 100
ckpt_every = 1000 
step = 0

model_optimizer = optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and not "lora_" in n]}, 
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n], 'lr': 2e-5}, 
    {'params': vqc.proj.parameters(), 'lr': 5e-4},
    {'params': vqc.post.parameters(), 'lr': 5e-4},
    {'params': vqc.n_qubits_to_vec_dim.parameters(), 'lr': 5e-4}
], lr=2e-5)

print("Starting training for Combined LoRA and VQC (Increased Qubits) model...")

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
            hidden = outputs.hidden_states[-1] 

            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)

            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction

            lm_logits = model.lm_head(hidden)

            loss_fct = torch.nn.CrossEntropyLoss()
            shift_logits = lm_logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        scaler.scale(loss).backward()

        if vqc.q_params.grad is not None:
             if not torch.isfinite(vqc.q_params.grad).all():
                 print(f"Step {step}: Warning: NaN or Inf gradient found for vqc.q_params. Skipping update.")
                 vqc.q_params.grad = None
             else:
                with torch.no_grad():
                    vqc.q_params -= 5e-4 * vqc.q_params.grad
                vqc.q_params.grad = None

        scaler.step(model_optimizer)
        scaler.update()

        step += 1
        if step % eval_every == 0:
            def evaluate_combined_qubits():
                model.eval() 
                total_loss = 0.0
                count = 0
                with torch.no_grad():
                    for batch in val_loader:
                        ids = batch["input_ids"].to(device)
                        att = batch["attention_mask"].to(device)
                        with torch.amp.autocast("cuda"):
                            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                            hidden = outputs.hidden_states[-1]
                            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                            lm_logits = model.lm_head(hidden)
                            loss_fct_eval = torch.nn.CrossEntropyLoss()
                            shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                            shift_labels_eval = ids[:, 1:].contiguous()
                            loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
                        total_loss += loss.item()
                        count += 1
                model.train()
                return total_loss / max(1, count)

            val_loss_combined_qubits = evaluate_combined_qubits()
            print(f"Combined Model (Increased Qubits) Step {step} Eval Loss: {val_loss_combined_qubits:.4f}")

        if step >= steps:
            break
    if step >= steps:
        break

def evaluate_combined_qubits():
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                hidden = outputs.hidden_states[-1]
                vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                lm_logits = model.lm_head(hidden)
                loss_fct_eval = torch.nn.CrossEntropyLoss()
                shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                shift_labels_eval = ids[:, 1:].contiguous()
                loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
            total_loss += loss.item()
            count += 1
    model.train()
    return total_loss / max(1, count)

val_loss_combined_qubits = evaluate_combined_qubits()
print(f"Combined LoRA and VQC (Increased Qubits) Training finished at step {step}. Final val loss: {val_loss_combined_qubits:.4f}")

trainable params: 294,912 || all params: 125,493,504 || trainable%: 0.2350
Loaded EleutherAI/gpt-neo-125M on cuda with LoRA adapter
Instantiated VQCAdapter with hidden_dim=768, n_qubits=4, vec_dim=16 on CPU
Starting training for Combined LoRA and VQC (Increased Qubits) model...


/tmp/ipython-input-2581211112.py:116: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Combined Model (Increased Qubits) Step 100 Eval Loss: 6.2221
Combined Model (Increased Qubits) Step 200 Eval Loss: 4.8849
Combined Model (Increased Qubits) Step 300 Eval Loss: 2.5639
Combined Model (Increased Qubits) Step 400 Eval Loss: 2.0461
Combined Model (Increased Qubits) Step 500 Eval Loss: 1.9846
Combined Model (Increased Qubits) Step 600 Eval Loss: 1.9562
Combined Model (Increased Qubits) Step 700 Eval Loss: 1.9374
Combined Model (Increased Qubits) Step 800 Eval Loss: 1.9229
Combined Model (Increased Qubits) Step 900 Eval Loss: 1.9108
Combined Model (Increased Qubits) Step 1000 Eval Loss: 1.9001
Combined Model (Increased Qubits) Step 1100 Eval Loss: 1.8908
Combined Model (Increased Qubits) Step 1200 Eval Loss: 1.8820
Combined Model (Increased Qubits) Step 1300 Eval Loss: 1.8744
Combined Model (Increased Qubits) Step 1400 Eval Loss: 1.8673
Combined Model (Increased Qubits) Step 1500 Eval Loss: 1.8506
Combined Model (Increased Qubits) Step 1600 Eval Loss: 1.8446
Combined Model (I

In [42]:
#Combined LoRA and VQC Model (4 Qubits, Modified Circuit, VQC LR 5e-4)

import os
import torch
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import pennylane as qml
from pennylane import numpy as pnp
import torch.nn as nn

class VQCAdapter(nn.Module):
    def __init__(self, hidden_dim, n_qubits=4, dev_name="default.qubit"):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_qubits = n_qubits
        self.vec_dim = 2 ** n_qubits
        self.proj = nn.Linear(hidden_dim, self.vec_dim)
        self.post = nn.Linear(self.vec_dim, hidden_dim)
        self.q_params = torch.nn.Parameter(torch.randn(self.n_qubits * 2 * 3))

        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.qnode = qml.QNode(self._circuit, self.dev, interface="torch")
        self.to_cpu = True
        self.n_qubits_to_vec_dim = torch.nn.Linear(self.n_qubits, self.vec_dim)


    def _circuit(self, inputs, qparams):
        inputs_scaled = inputs * torch.pi

        for i in range(self.n_qubits):
             qml.RY(inputs_scaled[i], wires=i)

        idx = 0
        for i in range(self.n_qubits):
            qml.RY(qparams[idx], wires=i); idx += 1
            qml.RZ(qparams[idx], wires=i); idx += 1

        for i in range(self.n_qubits - 1):
            qml.CZ(wires=[i, i+1])

        for i in range(self.n_qubits):
            qml.RY(qparams[idx], wires=i); idx += 1
            qml.RZ(qparams[idx], wires=i); idx += 1
            
        for i in range(self.n_qubits - 1):
            qml.CZ(wires=[i, i+1])

        for i in range(self.n_qubits):
            qml.RY(qparams[idx], wires=i); idx += 1
            qml.RZ(qparams[idx], wires=i); idx += 1

        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

    def forward(self, hidden):
        pass

def vqc_forward_cpu(hidden):
    vqc.proj.to("cpu")
    vqc.post.to("cpu")
    vqc.q_params = vqc.q_params.to("cpu")
    vqc.n_qubits_to_vec_dim.to("cpu")

    hid_cpu = hidden.detach().cpu()
    x = vqc.proj(hid_cpu)                       
    x = torch.tanh(x)
    outputs = []
    qparams = vqc.q_params.detach().cpu()
    for row in x:
        inp = row.view(-1)                       
        pooled = inp[:vqc.n_qubits].detach().numpy()
        res = torch.tensor(vqc.qnode(pooled, qparams.numpy()), dtype=torch.float32)
        outputs.append(res)
    out = torch.stack(outputs)                   
    mapped_out = vqc.n_qubits_to_vec_dim(out) 
    out = vqc.post(mapped_out)                          
    return out.to(hidden.device) 
    
MODEL_NAME = "EleutherAI/gpt-neo-125M"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM", 
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

model.train() 
print("Loaded", MODEL_NAME, "on", device, "with LoRA adapter")

hidden_dim = model.config.hidden_size
vqc = VQCAdapter(hidden_dim=hidden_dim, n_qubits=4) 

vqc.proj.to("cpu")
vqc.post.to("cpu")
vqc.q_params = vqc.q_params.to("cpu") 
vqc.n_qubits_to_vec_dim.to("cpu")

print(f"Instantiated VQCAdapter with hidden_dim={hidden_dim}, n_qubits={vqc.n_qubits} and modified circuit on CPU")

scaler = torch.cuda.amp.GradScaler()

steps = 2000
eval_every = 100
ckpt_every = 1000 
step = 0

model_optimizer = optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and not "lora_" in n]},
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n], 'lr': 2e-5}, 
    {'params': vqc.proj.parameters(), 'lr': 5e-4},
    {'params': vqc.post.parameters(), 'lr': 5e-4},
    {'params': vqc.n_qubits_to_vec_dim.parameters(), 'lr': 5e-4}
], lr=2e-5)


print("Starting training for Combined LoRA and VQC model with modified circuit...")

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]
            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction

            lm_logits = model.lm_head(hidden)
            loss_fct = torch.nn.CrossEntropyLoss()
            shift_logits = lm_logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        scaler.scale(loss).backward()

        if vqc.q_params.grad is not None:
             if not torch.isfinite(vqc.q_params.grad).all():
                 print(f"Step {step}: Warning: NaN or Inf gradient found for vqc.q_params. Skipping update.")
                 vqc.q_params.grad = None
             else:
                with torch.no_grad():
                    vqc.q_params -= 5e-4 * vqc.q_params.grad 
                vqc.q_params.grad = None

        scaler.step(model_optimizer)
        scaler.update()

        step += 1
        if step % eval_every == 0:
            def evaluate_combined():
                model.eval() 
                total_loss = 0.0
                count = 0
                with torch.no_grad():
                    for batch in val_loader:
                        ids = batch["input_ids"].to(device)
                        att = batch["attention_mask"].to(device)
                        with torch.amp.autocast("cuda"):
                            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                            hidden = outputs.hidden_states[-1]
                            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                            lm_logits = model.lm_head(hidden)
                            loss_fct_eval = torch.nn.CrossEntropyLoss()
                            shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                            shift_labels_eval = ids[:, 1:].contiguous()
                            loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))

                        total_loss += loss.item()
                        count += 1
                model.train()
                return total_loss / max(1, count)

            val_loss_combined_circuit = evaluate_combined()
            print(f"Combined Model (Modified Circuit) Step {step} Eval Loss: {val_loss_combined_circuit:.4f}")

        if step >= steps:
            break
    if step >= steps:
        break

def evaluate_combined():
    model.eval() 
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                hidden = outputs.hidden_states[-1]
                vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                lm_logits = model.lm_head(hidden)
                loss_fct_eval = torch.nn.CrossEntropyLoss()
                shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                shift_labels_eval = ids[:, 1:].contiguous()
                loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
            total_loss += loss.item()
            count += 1
    model.train()
    return total_loss / max(1, count)

val_loss_combined_circuit = evaluate_combined()
print(f"Combined LoRA and VQC Training (Modified Circuit) finished at step {step}. Final val loss: {val_loss_combined_circuit:.4f}")

trainable params: 294,912 || all params: 125,493,504 || trainable%: 0.2350
Loaded EleutherAI/gpt-neo-125M on cuda with LoRA adapter
Instantiated VQCAdapter with hidden_dim=768, n_qubits=4 and modified circuit on CPU
Starting training for Combined LoRA and VQC model with modified circuit...


/tmp/ipython-input-147848540.py:142: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Combined Model (Modified Circuit) Step 100 Eval Loss: 6.2028
Combined Model (Modified Circuit) Step 200 Eval Loss: 4.8044
Combined Model (Modified Circuit) Step 300 Eval Loss: 2.4190
Combined Model (Modified Circuit) Step 400 Eval Loss: 2.0562
Combined Model (Modified Circuit) Step 500 Eval Loss: 1.9875
Combined Model (Modified Circuit) Step 600 Eval Loss: 1.9573
Combined Model (Modified Circuit) Step 700 Eval Loss: 1.9383
Combined Model (Modified Circuit) Step 800 Eval Loss: 1.9233
Combined Model (Modified Circuit) Step 900 Eval Loss: 1.9116
Combined Model (Modified Circuit) Step 1000 Eval Loss: 1.9008
Combined Model (Modified Circuit) Step 1100 Eval Loss: 1.8911
Combined Model (Modified Circuit) Step 1200 Eval Loss: 1.8725
Combined Model (Modified Circuit) Step 1300 Eval Loss: 1.8448
Combined Model (Modified Circuit) Step 1400 Eval Loss: 1.8276
Combined Model (Modified Circuit) Step 1500 Eval Loss: 1.8208
Combined Model (Modified Circuit) Step 1600 Eval Loss: 1.8149
Combined Model (M

In [44]:
#Combined LoRA and VQC Model (4 Qubits, Modified Circuit, VQC LR 1e-3)
import os
import torch
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import pennylane as qml
from pennylane import numpy as pnp
import torch.nn as nn
import math

class VQCAdapter(nn.Module):
    def __init__(self, hidden_dim, n_qubits=4, dev_name="default.qubit", device_cpu=True): 
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_qubits = n_qubits
        self.vec_dim = 2 ** n_qubits
        self.proj = nn.Linear(hidden_dim, self.vec_dim)
        self.post = nn.Linear(self.vec_dim, hidden_dim)
        self.q_params = torch.nn.Parameter(torch.randn(self.n_qubits * 6)) 
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.qnode = qml.QNode(self._circuit, self.dev, interface="torch")
        self.to_cpu = device_cpu
        self.n_qubits_to_vec_dim = torch.nn.Linear(self.n_qubits, self.vec_dim)


    def _circuit(self, inputs, qparams):
        for i in range(self.n_qubits):
            qml.RY(inputs[i % len(inputs)], wires=i)
        idx = 0
        for i in range(self.n_qubits - 1):
            qml.CZ(wires=[i, i+1])
        for i in range(self.n_qubits):
             qml.RY(qparams[idx], wires=i); idx += 1
        for i in range(self.n_qubits - 1):
            qml.CZ(wires=[i, i+1])
        for i in range(self.n_qubits):
             qml.RZ(qparams[idx], wires=i); idx += 1

        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]


def vqc_forward_cpu(hidden):
    vqc.proj.to("cpu")
    vqc.post.to("cpu")
    vqc.q_params = vqc.q_params.to("cpu") 
    vqc.n_qubits_to_vec_dim.to("cpu")


    hid_cpu = hidden.detach().cpu()
    x = vqc.proj(hid_cpu)                  
    x = torch.tanh(x)
    outputs = []
    qparams = vqc.q_params.detach().cpu()
    for row in x:
        inp = row.view(-1)
        pooled = inp[:vqc.n_qubits].detach().numpy()
        res = torch.tensor(vqc.qnode(pooled, qparams.numpy()), dtype=torch.float32)
        outputs.append(res)
    out = torch.stack(outputs)
    mapped_out = vqc.n_qubits_to_vec_dim(out)
    out = vqc.post(mapped_out)                          
    return out.to(hidden.device) 


MODEL_NAME = "EleutherAI/gpt-neo-125M"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05,
    bias="none", 
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

model.train() 
print("Loaded", MODEL_NAME, "on", device, "with LoRA adapter")

hidden_dim = model.config.hidden_size

vqc = VQCAdapter(hidden_dim=hidden_dim, n_qubits=4)

vqc.proj.to("cpu")
vqc.post.to("cpu")
vqc.q_params = vqc.q_params.to("cpu")
vqc.n_qubits_to_vec_dim.to("cpu")
print(f"Instantiated VQCAdapter with hidden_dim={hidden_dim}, n_qubits={vqc.n_qubits} on CPU and modified circuit")

scaler = torch.cuda.amp.GradScaler()

steps = 2000
eval_every = 100
ckpt_every = 1000
step = 0

new_vqc_lr = 1e-3 

model_optimizer = optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and not "lora_" in n]}, 
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n], 'lr': 2e-5}, 
    {'params': vqc.proj.parameters(), 'lr': new_vqc_lr}, 
    {'params': vqc.post.parameters(), 'lr': new_vqc_lr}, 
    {'params': vqc.n_qubits_to_vec_dim.parameters(), 'lr': new_vqc_lr} 
], lr=2e-5) 

print(f"Starting training for Combined LoRA and VQC model with VQC LR = {new_vqc_lr}...")

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]

            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)

            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction

            lm_logits = model.lm_head(hidden)

            loss_fct = torch.nn.CrossEntropyLoss()
            shift_logits = lm_logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        scaler.scale(loss).backward()

        if vqc.q_params.grad is not None:
             if not torch.isfinite(vqc.q_params.grad).all():
                 print(f"Step {step}: Warning: NaN or Inf gradient found for vqc.q_params. Skipping update.")
                 vqc.q_params.grad = None
             else:
                with torch.no_grad():
                    vqc.q_params -= new_vqc_lr * vqc.q_params.grad
                vqc.q_params.grad = None

        scaler.step(model_optimizer)
        scaler.update()

        step += 1
        if step % eval_every == 0:
            def evaluate_combined():
                model.eval()
                total_loss = 0.0
                count = 0
                with torch.no_grad():
                    for batch in val_loader:
                        ids = batch["input_ids"].to(device)
                        att = batch["attention_mask"].to(device)
                        with torch.amp.autocast("cuda"):
                            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                            hidden = outputs.hidden_states[-1]
                            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                            lm_logits = model.lm_head(hidden)
                            loss_fct_eval = torch.nn.CrossEntropyLoss()
                            shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                            shift_labels_eval = ids[:, 1:].contiguous()
                            loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
                        total_loss += loss.item()
                        count += 1
                model.train()
                return total_loss / max(1, count)

            val_loss_combined_lr = evaluate_combined()
            print(f"Combined Model (VQC LR={new_vqc_lr}) Step {step} Eval Loss: {val_loss_combined_lr:.4f}")

        if step >= steps:
            break
    if step >= steps:
        break

def evaluate_combined():
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                hidden = outputs.hidden_states[-1]
                vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                lm_logits = model.lm_head(hidden)
                loss_fct_eval = torch.nn.CrossEntropyLoss()
                shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                shift_labels_eval = ids[:, 1:].contiguous()
                loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
            total_loss += loss.item()
            count += 1
    model.train()
    return total_loss / max(1, count)

val_loss_combined_lr = evaluate_combined()
combined_perplexity_lr = math.exp(val_loss_combined_lr)

print(f"Combined LoRA and VQC Training (VQC LR={new_vqc_lr}) finished at step {step}. Final val loss: {val_loss_combined_lr:.4f}")
print(f"Combined LoRA and VQC Model (VQC LR={new_vqc_lr}) Perplexity: {combined_perplexity_lr:.4f}")

Loaded EleutherAI/gpt-neo-125M on cuda with LoRA adapter
Instantiated VQCAdapter with hidden_dim=768, n_qubits=4 on CPU and modified circuit
Starting training for Combined LoRA and VQC model with VQC LR = 0.001...


/tmp/ipython-input-156396673.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Combined Model (VQC LR=0.001) Step 100 Eval Loss: 6.2640
Combined Model (VQC LR=0.001) Step 200 Eval Loss: 4.9671
Combined Model (VQC LR=0.001) Step 300 Eval Loss: 2.6015
Combined Model (VQC LR=0.001) Step 400 Eval Loss: 2.0604
Combined Model (VQC LR=0.001) Step 500 Eval Loss: 1.9909
Combined Model (VQC LR=0.001) Step 600 Eval Loss: 1.9595
Combined Model (VQC LR=0.001) Step 700 Eval Loss: 1.9400
Combined Model (VQC LR=0.001) Step 800 Eval Loss: 1.9253
Combined Model (VQC LR=0.001) Step 900 Eval Loss: 1.9130
Combined Model (VQC LR=0.001) Step 1000 Eval Loss: 1.9023
Combined Model (VQC LR=0.001) Step 1100 Eval Loss: 1.8923
Combined Model (VQC LR=0.001) Step 1200 Eval Loss: 1.8838
Combined Model (VQC LR=0.001) Step 1300 Eval Loss: 1.8662
Combined Model (VQC LR=0.001) Step 1400 Eval Loss: 1.8490
Combined Model (VQC LR=0.001) Step 1500 Eval Loss: 1.8427
Combined Model (VQC LR=0.001) Step 1600 Eval Loss: 1.8368
Combined Model (VQC LR=0.001) Step 1700 Eval Loss: 1.8307
Combined Model (VQC LR=

In [46]:
#Combined LoRA and VQC Model (4 Qubits, Modified Circuit, VQC LR 1e-4)

import os
import torch
from tqdm import tqdm
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
import pennylane as qml
from pennylane import numpy as pnp
import torch.nn as nn
import math

class VQCAdapter(nn.Module):
    def __init__(self, hidden_dim, n_qubits=4, dev_name="default.qubit", device_cpu=True): 
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_qubits = n_qubits
        self.vec_dim = 2 ** n_qubits
        self.proj = nn.Linear(hidden_dim, self.vec_dim)
        self.post = nn.Linear(self.vec_dim, hidden_dim)
        self.q_params = torch.nn.Parameter(torch.randn(self.n_qubits * 6)) 
        self.dev = qml.device("default.qubit", wires=self.n_qubits)
        self.qnode = qml.QNode(self._circuit, self.dev, interface="torch")
        self.to_cpu = device_cpu
        self.n_qubits_to_vec_dim = torch.nn.Linear(self.n_qubits, self.vec_dim)


    def _circuit(self, inputs, qparams):
        for i in range(self.n_qubits):
            qml.RY(inputs[i % len(inputs)], wires=i)
        idx = 0
        for i in range(self.n_qubits - 1):
            qml.CZ(wires=[i, i+1])
        for i in range(self.n_qubits):
             qml.RY(qparams[idx], wires=i); idx += 1
        for i in range(self.n_qubits - 1):
            qml.CZ(wires=[i, i+1])
        for i in range(self.n_qubits):
             qml.RZ(qparams[idx], wires=i); idx += 1

        return [qml.expval(qml.PauliZ(i)) for i in range(self.n_qubits)]

def vqc_forward_cpu(hidden):
    vqc.proj.to("cpu")
    vqc.post.to("cpu")
    vqc.q_params = vqc.q_params.to("cpu") 
    vqc.n_qubits_to_vec_dim.to("cpu")


    hid_cpu = hidden.detach().cpu()
    x = vqc.proj(hid_cpu)                       
    x = torch.tanh(x)
    outputs = []
    qparams = vqc.q_params.detach().cpu()
    for row in x:
        inp = row.view(-1)                      
        pooled = inp[:vqc.n_qubits].detach().numpy()
        res = torch.tensor(vqc.qnode(pooled, qparams.numpy()), dtype=torch.float32)
        outputs.append(res)
    out = torch.stack(outputs)
    mapped_out = vqc.n_qubits_to_vec_dim(out)
    out = vqc.post(mapped_out)                          
    return out.to(hidden.device)


MODEL_NAME = "EleutherAI/gpt-neo-125M"
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["q_proj", "v_proj"], 
    lora_dropout=0.05, 
    bias="none", 
    task_type="CAUSAL_LM", 
)
model = get_peft_model(model, lora_config)

model.train() 
print("Loaded", MODEL_NAME, "on", device, "with LoRA adapter")

hidden_dim = model.config.hidden_size

vqc = VQCAdapter(hidden_dim=hidden_dim, n_qubits=4)

vqc.proj.to("cpu")
vqc.post.to("cpu")
vqc.q_params = vqc.q_params.to("cpu")
vqc.n_qubits_to_vec_dim.to("cpu")
print(f"Instantiated VQCAdapter with hidden_dim={hidden_dim}, n_qubits={vqc.n_qubits} on CPU and modified circuit")

scaler = torch.cuda.amp.GradScaler()

steps = 2000
eval_every = 100
ckpt_every = 1000
step = 0

new_vqc_lr = 1e-4 

model_optimizer = optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and not "lora_" in n]}, 
    {'params': [p for n, p in model.named_parameters() if p.requires_grad and "lora_" in n], 'lr': 2e-5}, 
    {'params': vqc.proj.parameters(), 'lr': new_vqc_lr}, 
    {'params': vqc.post.parameters(), 'lr': new_vqc_lr}, 
    {'params': vqc.n_qubits_to_vec_dim.parameters(), 'lr': new_vqc_lr} 
], lr=2e-5) 


print(f"Starting training for Combined LoRA and VQC model with VQC LR = {new_vqc_lr}...")

for epoch in range(1000):
    for batch in train_loader:
        ids = batch["input_ids"].to(device)
        att = batch["attention_mask"].to(device)

        model_optimizer.zero_grad()

        with torch.amp.autocast("cuda"):
            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]

            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)

            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction

            lm_logits = model.lm_head(hidden)

            loss_fct = torch.nn.CrossEntropyLoss()
            shift_logits = lm_logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

        scaler.scale(loss).backward()

        if vqc.q_params.grad is not None:
             if not torch.isfinite(vqc.q_params.grad).all():
                 print(f"Step {step}: Warning: NaN or Inf gradient found for vqc.q_params. Skipping update.")
                 vqc.q_params.grad = None
             else:
                with torch.no_grad():
                    vqc.q_params -= new_vqc_lr * vqc.q_params.grad
                vqc.q_params.grad = None

        scaler.step(model_optimizer)
        scaler.update()

        step += 1
        if step % eval_every == 0:
            def evaluate_combined():
                model.eval()
                total_loss = 0.0
                count = 0
                with torch.no_grad():
                    for batch in val_loader:
                        ids = batch["input_ids"].to(device)
                        att = batch["attention_mask"].to(device)
                        with torch.amp.autocast("cuda"):
                            outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                            hidden = outputs.hidden_states[-1]
                            vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                            hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                            lm_logits = model.lm_head(hidden)
                            loss_fct_eval = torch.nn.CrossEntropyLoss()
                            shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                            shift_labels_eval = ids[:, 1:].contiguous()
                            loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
                        total_loss += loss.item()
                        count += 1
                model.train()
                return total_loss / max(1, count)

            val_loss_combined_lr_lower = evaluate_combined()
            print(f"Combined Model (VQC LR={new_vqc_lr}) Step {step} Eval Loss: {val_loss_combined_lr_lower:.4f}")

        if step >= steps:
            break
    if step >= steps:
        break

def evaluate_combined():
    model.eval()
    total_loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in val_loader:
            ids = batch["input_ids"].to(device)
            att = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                outputs = model(input_ids=ids, attention_mask=att, output_hidden_states=True)
                hidden = outputs.hidden_states[-1]
                vqc_correction = vqc_forward_cpu(hidden[:, -1, :].float()).to(hidden.dtype)
                hidden[:, -1, :] = hidden[:, -1, :] + vqc_correction
                lm_logits = model.lm_head(hidden)
                loss_fct_eval = torch.nn.CrossEntropyLoss()
                shift_logits_eval = lm_logits[:, :-1, :].contiguous()
                shift_labels_eval = ids[:, 1:].contiguous()
                loss = loss_fct_eval(shift_logits_eval.view(-1, shift_logits_eval.size(-1)), shift_labels_eval.view(-1))
            total_loss += loss.item()
            count += 1
    model.train()
    return total_loss / max(1, count)

val_loss_combined_lr_lower = evaluate_combined()
combined_perplexity_lr_lower = math.exp(val_loss_combined_lr_lower)

print(f"Combined LoRA and VQC Training (VQC LR={new_vqc_lr}) finished at step {step}. Final val loss: {val_loss_combined_lr_lower:.4f}")
print(f"Combined LoRA and VQC Model (VQC LR={new_vqc_lr}) Perplexity: {combined_perplexity_lr_lower:.4f}")

Loaded EleutherAI/gpt-neo-125M on cuda with LoRA adapter
Instantiated VQCAdapter with hidden_dim=768, n_qubits=4 on CPU and modified circuit
Starting training for Combined LoRA and VQC model with VQC LR = 0.0001...


/tmp/ipython-input-1140352663.py:114: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


Combined Model (VQC LR=0.0001) Step 100 Eval Loss: 6.2452
Combined Model (VQC LR=0.0001) Step 200 Eval Loss: 4.9064
Combined Model (VQC LR=0.0001) Step 300 Eval Loss: 2.4238
Combined Model (VQC LR=0.0001) Step 400 Eval Loss: 2.0790
Combined Model (VQC LR=0.0001) Step 500 Eval Loss: 2.0026
Combined Model (VQC LR=0.0001) Step 600 Eval Loss: 1.9688
Combined Model (VQC LR=0.0001) Step 700 Eval Loss: 1.9475
Combined Model (VQC LR=0.0001) Step 800 Eval Loss: 1.9320
Combined Model (VQC LR=0.0001) Step 900 Eval Loss: 1.9179
Combined Model (VQC LR=0.0001) Step 1000 Eval Loss: 1.9069
Combined Model (VQC LR=0.0001) Step 1100 Eval Loss: 1.8969
Combined Model (VQC LR=0.0001) Step 1200 Eval Loss: 1.8883
Combined Model (VQC LR=0.0001) Step 1300 Eval Loss: 1.8803
Combined Model (VQC LR=0.0001) Step 1400 Eval Loss: 1.8729
Combined Model (VQC LR=0.0001) Step 1500 Eval Loss: 1.8662
Combined Model (VQC LR=0.0001) Step 1600 Eval Loss: 1.8597
Combined Model (VQC LR=0.0001) Step 1700 Eval Loss: 1.8533
Combin

In [45]:
print("--- Model Perplexity Comparison (Including Different VQC LR) ---")
print(f"VQC Model Perplexity (3 Qubits, Original Circuit): {vqc_perplexity:.4f}")
print(f"No VQC Baseline Perplexity: {no_vqc_perplexity:.4f}")
print(f"LoRA Baseline Perplexity: {lora_perplexity:.4f}")
print(f"Combined LoRA and VQC Model Perplexity (3 Qubits, Original Circuit): {combined_perplexity:.4f}")
print(f"Combined LoRA and VQC Model Perplexity (4 Qubits, Original Circuit): {combined_perplexity_qubits:.4f}")
print(f"Combined LoRA and VQC Model Perplexity (4 Qubits, Modified Circuit, VQC LR 5e-4): {combined_perplexity_circuit:.4f}")
print(f"Combined LoRA and VQC Model Perplexity (4 Qubits, Modified Circuit, VQC LR 1e-3): {combined_perplexity_lr:.4f}")
print(f"Combined LoRA and VQC Model Perplexity (4 Qubits, Modified Circuit, VQC LR 1e-4): {combined_perplexity_lr_lower:.4f}")

--- Model Perplexity Comparison (Including Different VQC LR) ---
VQC Model Perplexity (3 Qubits, Original Circuit): 7.7600
No VQC Baseline Perplexity: 6.3996
LoRA Baseline Perplexity: 6.2658
Combined LoRA and VQC Model Perplexity (3 Qubits, Original Circuit): 6.2661
Combined LoRA and VQC Model Perplexity (4 Qubits, Original Circuit): 6.1349
Combined LoRA and VQC Model Perplexity (4 Qubits, Modified Circuit, VQC LR 5e-4): 5.8428
Combined LoRA and VQC Model Perplexity (4 Qubits, Modified Circuit, VQC LR 1e-3): 6.0666
Combined LoRA and VQC Model Perplexity (4 Qubits, Modified Circuit, VQC LR 1e-4): 6.2800



# Summary:

*   Comparing the final validation perplexity:
    *   VQC-only Model (3 Qubits, Original Circuit): **7.7600**
    *   No VQC Baseline: **6.3996**
    *   LoRA-only Baseline: **6.2658**
    *   Combined LoRA and VQC Model (3 Qubits, Original Circuit, VQC LR 5e-4): **6.2661**
    *   Combined LoRA and VQC Model (4 Qubits, Original Circuit, VQC LR 5e-4): **6.1349**
    *   Combined LoRA and VQC Model (4 Qubits, Modified Circuit, VQC LR 5e-4): **5.8428** (Best performance)
    *   Combined LoRA and VQC Model (4 Qubits, Modified Circuit, VQC LR 1e-3): **6.0666**
    *   Combined LoRA and VQC Model (4 Qubits, Modified Circuit, VQC LR 1e-4): **6.2800**

*   The best-performing model was the **Combined LoRA and VQC Model with 4 qubits and the modified circuit structure (VQC LR 5e-4)**, achieving a perplexity of **5.8428**.
*   This represents a **6.7495%**, almost **7%** improvement in perplexity compared to the LoRA-only baseline (6.2658).